In [49]:
import json as js
import re
from pathlib import Path


In [50]:
# Load Logic-Based Data
input_path = Path("logic_based.json")
with input_path.open("r", encoding="utf-8") as f:
    logic_based_data = js.load(f)

print(len(logic_based_data))

411


In [51]:
# Count samples missing premises-FOL
missing_fol = sum(1 for sample in logic_based_data if not sample.get("premises-FOL"))
missing_fol

2

In [ ]:
# normalize FOL strings and save JSON to data/processed
def normalize_fol_entry(text: str) -> str:
    if not text:
        return text
    # clean corrupted ascii quantifiers like "�forall" / "�exists"
    text = re.sub(r"�\s*forall\b", "forall", text, flags=re.IGNORECASE)
    text = re.sub(r"�\s*exists\b", "exists", text, flags=re.IGNORECASE)
    # normalize ascii quantifiers with function-like or comma syntax
    text = re.sub(r"\bforall\s*\(\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*,\s*", r"ForAll(\1, ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bexists\s*\(\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*,\s*", r"Exists(\1, ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bforall\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*", r"ForAll(\1, ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bexists\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*", r"Exists(\1, ", text, flags=re.IGNORECASE)
    # normalize unicode quantifiers with function-like or comma syntax
    text = re.sub(r"∀\s*\(\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*,\s*", r"ForAll(\1, ", text)
    text = re.sub(r"∃\s*\(\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*,\s*", r"Exists(\1, ", text)
    text = re.sub(r"∀\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*,\s*", r"ForAll(\1, ", text)
    text = re.sub(r"∃\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*,\s*", r"Exists(\1, ", text)
    text = re.sub(r"∀\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*", r"ForAll(\1, ", text)
    text = re.sub(r"∃\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*", r"Exists(\1, ", text)
    # remaining symbol replacements
    replacements = {
        "→": "->",
        "∧": "AND",
        "∨": "OR",
        "¬": "NOT ",
        "↔": "<->",
        "≥": ">=",
        "≤": "<=",
        "≠": "!=",
        "∈": "IN",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

processed_data = []
for sample in logic_based_data:
    new_sample = dict(sample)
    fol_list = sample.get("premises-FOL")
    if fol_list is not None:
        new_sample["premises-FOL"] = [normalize_fol_entry(p) for p in fol_list]
    processed_data.append(new_sample)

output_path = Path("processed/logic_based.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    js.dump(processed_data, f, ensure_ascii=False, indent=2)

output_path.as_posix()

'processed/logic_based.json'

In [53]:
processed_data[0]["premises-FOL"][:3]

['ForAll(x, (WT(x) -> O(x))',
 'ForAll(x, (NOT PEP8(x) -> NOT WT(x))',
 'ForAll(x, (EM(x))']

In [54]:
# unique characters after cleanup
unique_chars = set("".join("".join(s.get("premises-FOL", [])) for s in processed_data))
print(unique_chars)

{',', 'à', 'A', 'X', '3', 'x', 'C', 'O', 'n', '6', 'o', '^', '2', '1', '+', 'b', '4', ')', 'z', '%', '_', '8', 'V', 'w', ':', 'd', 'M', 'Q', 'h', 'Y', '<', 'c', 'U', 'k', 'j', 'F', 'P', 'H', 't', 'p', '>', 'q', 'S', 's', 'J', 'g', 'a', '5', '&', 'y', '0', '-', ' ', 'L', '}', 'I', '7', 'N', 'G', '!', 'K', 'B', 'l', '=', '9', "'", 'R', '~', '*', '∈', '(', 'r', '/', 'i', 'm', 'T', '{', 'W', 'u', '.', 'v', 'D', 'E', 'e', 'f', 'Z', 'â'}
